<div style="background:#C1ABA6">
<div style="font-size: xx-large ; font-weight: 900 ;  padding-top: 100px ; color: rgba(0 , 0 , 0 , 0.8); line-height: 100%"; align="center">Moja prva karta</div>
    
---
<div align="center"> Vježbe iz Seizmologije I </div>
<div align="center"> ak. god. 2026./2027. </div>
<div align="center"> dr.sc. Katarina Zailac </div>

---
</div>

# PyGMT

PyGMT je Python wrapper za [Generic Mapping Tool (GMT)](https://www.generic-mapping-tools.org/), program koji se koristi u Linux terminalu s mogućnostima obrade prostornih podataka i izrade kvalitetnih slika. Primjer jedne takve skripte je u nastavku.

```bash
#!/bin/bash
name='figure'

bounds='-R11.0/22/40.9/48.0'
proj='-JM11c'

gmt begin $name pdf
       # Generate topography image with shading
        gmt grdimage @earth_relief_30s -Cwhite,darkgray $bounds $proj -I+d

        # Overlay basemap
        gmt coast $bounds $proj -Bpx4f2 -Bpy2 -BWneS -Dh -N1 --FONT_LABEL=10p,Helvetica,black --FONT_ANNOT=10p,Helvetica,black
gmt end show
```

GMT je vrlo moćan alat, ali ima jako strmu krivulju učenja, pogotovo za korisnike koji nisu navikli na rad u terminalu. Stoga je nastao [PyGMT](https://www.pygmt.org/latest/), koji je puno jednostavniji za korištenje, a ima sve mogućnosti GMT-a.

PyGMT je zapravo Python paket, pa se kao svi paketi mora importati na početku skripte. Sve metode potrebne za generiranje slika dostupne su iz krovnog paketa.

In [ ]:
import pygmt

# Osnove

## Postavljanje granica karte

Nacrtajmo obalu Europskog kontinenta. Za to ćemo koristiti metodu [`pygmt.Figure.coast()`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.coast.html#pygmt.Figure.coast) koja crta kontinente, države, obale i rijeke.

Granicu karte koju želimo prikazati zadajemo parametrom `region`.

### Koordinate

Regiju možemo zadati u obliku varijable tipa string u formatu `xmin/xmax/ymin/ymax`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    # postavljamo granice karte
    region="-12.5/50/33/72",
    # Mercator projekcija sa širinom karte od 15 cm
    projection="M15c",
    # postavimo da se nacrtaju obale
    shorelines=True
)

fig.show()

Regija se može zadati i u obliku liste koordinata, u kojem slučaju one moraju biti zadane kao `[xmin, xmax, ymin, ymax]`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True
)

fig.show()

Osim kao granice osi, regija se može zadati i zadavanjem koordinata donjeg lijevog kuta i gornjeg desnog kuta. U tom slučaju se na string koordinata dodaje tekst `+r`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region="-12/33/50/72+r",
    projection="M15c",
    shorelines=True
)

fig.show()

### Globalne regije

U slučaju karata koje pokrivaju cijeli planet, postoje dvije opcije. Ako se koristi argument `d`, karta će prikazivati svijet u rasponu koordinata od -180° do 180° te -90° do 90°. Karta je centrirana na sjecištu ekvatora i nultog meridijana.

In [ ]:
fig = pygmt.Figure()
fig.coast(
    region="d",
    projection="Cyl_stere/12c",
    shorelines=True
)
fig.show()

Ako se koristi argument `g`, koordinate su u rasponu od 0° do 360° te od -90° do 90°. Karta je centrirana u sjecištu ekvatora i međunarodne datumske granice.

In [ ]:
fig = pygmt.Figure()
fig.coast(
    region="g",
    projection="Cyl_stere/12c",
    shorelines=True
)
fig.show()

### ISO kod

Regija se također može zadati korištenjem [ISO 3166-1 alpha-2](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2) konvencije. Tako bi se regija RH nacrtala korištenjem koda `HR`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region="HR",
    projection="M15c",
    shorelines=True
)

fig.show()

Granica definirana ISO kodom se može proširiti dodavanjem `+r<inkrement>` na ISO kod.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region="HR+r1",
    projection="M15c",
    shorelines=True
)

fig.show()

`<inkrement>` se može zadati i različito u dva smjera, u kojem slučaju je format onda `xinc/yinc`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region="HR+r1/2",
    projection="M15c",
    shorelines=True
)

fig.show()

## Državne granice

Parametrom `borders` u metodi [`pygmt.Figure.coast()`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.coast.html#pygmt.Figure.coast) se definiraju političke granice koje će se prikazati na karti te se definira linija kakvom se žele označiti. Moguće razine koje se mogu koristiti:

* 1 - nacionalne granice
* 2 - granice pojedinih saveznih država u Sjevernoj i Južnoj Americi
* 3 - morske granice
* a - sve gore navedene granice (1 - 3).

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region="HR",
    projection="M15c",
    shorelines=True,
    borders="a"
)

fig.show()

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines="1.5p,blue",  # isto kao i za granice
    borders="a/2p,red"
)

fig.show()

## Boja kopna i vodenih površina

Za definiciju boja kopna i vodenih površina koriste se parametri `land` i `water`. Za boje se može koristiti [heksadecimalne kodove boja](https://www.color-hex.com/) ili [predefinirane nazive](https://docs.generic-mapping-tools.org/6.4/gmtcolors.html).

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True,
    borders=1,
    land="lightgray",
    water="lightblue"
)

fig.show()

### Okvir slike, mreža, naslov

#### Okvir

Kako bi se dodao lijepi okvir na sliku, koristi se parametar `frame`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True,
    borders=1,
    land="lightgray",
    water="lightblue",
    frame=True
)

fig.show()

#### Mreža

Zadavanjem parametra `frame=True` ili `frame="af"`, bit će označene oznake koordinata na intervalima koje odredi GMT (`a` od **a**notations). Ako želimo imati označenu i mrežu, to ćemo postići dodavanjem i `g` u parametar `frame`.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True,
    borders=1,
    land="lightgray",
    water="lightblue",
    frame="ag"
)

fig.show()

Intervali za oznake, okvir i mrežu se mogu dodati dopisivanjem željenog koraka iza `a`, `f` i `g`. U primjeru su ti koraci definirani kao 10° za oznake, 5° za okvir i 15° za mrežu.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True,
    borders=1,
    land="lightgray",
    water="lightblue",
    frame="a10f5g15"
)

fig.show()

#### Naslov

Naslov na sliku se dodaje dodavanjem `+t<naslov>` na parametar `frame`. Više opcija za parametar `frame` se onda dodaje u obliku liste.

<div class="alert alert-warning rounded-pill rounded-5" style="margin: auto auto 10px auto; text-align: center;">
    <strong>Napomena</strong>: U novijim verzijama <a href=https://www.pygmt.org/latest/>PyGMT-a</a> ovo se promijenilo. Uvijek provjerite dokumentaciju za verziju koju imate instaliranu!
</div>

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-12.5, 50, 33, 72],
    projection="M15c",
    shorelines=True,
    borders=1,
    land="lightgray",
    water="lightblue",
    frame=["a10f5g15", "+tKarta Europe"]
)

fig.show()

# Zadatak 1

Nacrtajte kartu Hrvatske. Odaberite neku boje za kopno i more, dodajte naslov i neka se na karti vidi mreža.

# Karta s reljefom

Najjednostavnije je koristiti metodu [`load_earth_relief()`](https://www.pygmt.org/dev/api/generated/pygmt.datasets.load_earth_relief.html).

<div class="alert alert-warning rounded-pill rounded-5" style="margin: auto auto 10px auto; text-align: center;">
    <strong>Napomena</strong>: Kod prvog dohvaćanja ovaj korak može potrajati!
</div>

In [ ]:
grid = pygmt.datasets.load_earth_relief(
    region="HR",
    resolution="01m"
)

Za crtanje podloge koristi se metoda [`pygmt.Figure.grdimage()`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.grdimage.html#pygmt.Figure.grdimage). Za prikaz ćemo koristiti predefiniranu skalu boja `geo`. Lista svih dostupnih skala je dostupna na stranicama alata [GMT](https://docs.generic-mapping-tools.org/6.5/reference/cpts.html).

Metoda [`pygmt.Figure.grdimage()`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.grdimage.html#pygmt.Figure.grdimage) prihvaća standardne parametre za crtanje (`region`, `projection`, `frame`).

In [ ]:
fig = pygmt.Figure()

projection = "M15c"  # Mercator projekcija

fig.grdimage(
    grid=grid,
    region="HR",
    projection=projection,
    cmap="geo",
    frame=True
)

fig.show()

Budući da [PyGMT](https://www.pygmt.org/latest/) dodaje dodatne elemente poput slojeva na postojeću instancu slike (ovdje označeno varijablom `fig`), kako bismo dodali granice na sliku, možemo samo dodati poziv metode [`pygmt.Figure.coast()`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.coast.html#pygmt.Figure.coast).

<div class="alert alert-info rounded-pill rounded-5" style="margin: auto auto 10px auto; text-align: center;">
    Kod poziva metode <a href=https://www.pygmt.org/dev/api/generated/pygmt.Figure.coast.html#pygmt.Figure.coast>coast()</a>, nije više potrebno definirati parametre koji su zadani ranije.
</div>

In [ ]:
fig = pygmt.Figure()

projection = "M15c"  # Mercator projekcija

fig.grdimage(
    grid=grid,
    region="HR",
    projection=projection,
    cmap="geo",
    frame=True
)

fig.coast(
    borders=1,
    shorelines=True
)

fig.show()

# Ucrtavanje podataka na kartu

Koristit ćemo skup podataka koji se instalira zajedno s GMT-jem. Do tih podataka se dolazi pomoću funkcije [`pygmt.datasets.load_sample_data`](https://www.pygmt.org/latest/api/generated/pygmt.datasets.load_sample_data.html). Učitat ćemo skup podataka o potresima u Japanu koji su uzrokovali tsunamije.

In [ ]:
data = pygmt.datasets.load_sample_data(name="japan_quakes")
data.head()

Postavit ćemo da je područje malo veće od područja u kojem postoje podaci.

In [ ]:
region = [
    data.longitude.min() - 1,
    data.longitude.max() + 1,
    data.latitude.min() - 1,
    data.latitude.max() + 1,
]

In [ ]:
fig = pygmt.Figure()
fig.coast(
    region=region, 
    projection="M15c", 
    frame=True, 
    land="black", 
    water="skyblue"
)
fig.plot(
    x=data.longitude, 
    y=data.latitude, 
    style="c0.3c", 
    fill="white", 
    pen="black"
)
fig.show()

Parametar `style` je bio zadan kao `c0.3c`, što znači da se koriste krugovi (`circle`) promjera 0.3 cm. Lista definiranih simbola je dana na [stranici](https://www.pygmt.org/v0.16.0/gallery/symbols/basic_symbols.html).

Parametar `pen` definira boju ruba simbola, a `fill` definira boju ispune.

Možemo skalirati veličinu kružnica s magnitudom potresa, tako da zadamo niz u parametar `size`.

In [ ]:
fig = pygmt.Figure()
fig.coast(
    region=region, 
    projection="M15c", 
    frame=True, 
    land="black", 
    water="skyblue"
)
fig.plot(
    x=data.longitude, 
    y=data.latitude,
    size=0.02 * (2**data.magnitude),
    style="cc",  # sada ne definiramo veličinu ovdje, samo tip simbola i mjernu jedinicu
    fill="white", 
    pen="black"
)
fig.show()

Također možemo mapirati i dubinu potresa na kartu, tako da dubine pojedinih hipocentara zadamo u parametar `fill`. Potrebno je definirati i skalu boja koja će se koristiti. Da bi se skala koristila, potrebno je zadati parametar `cmap=True` u `plot()` metodi.

In [ ]:
fig = pygmt.Figure()

pygmt.makecpt(cmap="viridis", series=[data.depth_km.min(), data.depth_km.max()])

fig.coast(
    region=region, 
    projection="M15c", 
    frame=True, 
    land="black", 
    water="skyblue"
)

fig.plot(
    x=data.longitude, 
    y=data.latitude,
    size=0.02 * (2**data.magnitude),
    style="cc",  # sada ne definiramo veličinu ovdje, samo tip simbola i mjernu jedinicu
    fill=data.depth_km, 
    pen="black",
    cmap=True
)

fig.colorbar(frame="xaf+lDubina (km)")

fig.show()

# Zadatak 2

Učitajte katalog potresa u datoteci `./data/2020_events.xml`. Filtrirajte katalog tako da koristite samo potrese magnitude veće ili jednake 3.5. Skalirajte veličinu kružnica koje označavaju epicentre s magnitudom potresa, a boja neka označava dubinu hipocentra. Neka karta bude topografska. Dodajte i naslov na kartu. Koristite neku drugu skalu boja.